# Scrubbing PII from LLM Conversation Logs

This notebook demonstrates **Substitute** mode on multi-turn conversation logs
(OpenAI-format JSONL). The pipeline:

1. Load JSONL, explore the data
2. Serialize each conversation into a single text (all roles), write to CSV
3. Run Anonymizer with `Substitute()` to replace detected PII with synthetic values
4. Inspect detected entities and substitutions
5. Split scrubbed text back into messages and write out as JSONL

## 1. Load and explore

In [1]:
import json
from pathlib import Path

import pandas as pd

JSONL_PATH = Path("reduced.redacted.jsonl")

conversations = []
with open(JSONL_PATH) as f:
    for line in f:
        conversations.append(json.loads(line))

print(f"{len(conversations)} conversations loaded")

12 conversations loaded


In [2]:
for i, conv in enumerate(conversations):
    msgs = conv["messages"]
    roles = sorted(set(m["role"] for m in msgs))
    total_chars = sum(len(m.get("content") or "") for m in msgs)
    print(
        f"  [{i:2d}] {len(msgs):3d} msgs | {total_chars:>7,} chars | "
        f"roles={roles} | cats={conv.get('categories', [])}"
    )

  [ 0]  25 msgs |  71,564 chars | roles=['assistant', 'system', 'tool', 'user'] | cats=['tool_heavy', 'multi_turn_complex']
  [ 1]   2 msgs |   2,309 chars | roles=['system', 'user'] | cats=['dissatisfied']
  [ 2]   3 msgs |  10,028 chars | roles=['system', 'user'] | cats=['dissatisfied']
  [ 3]  16 msgs | 164,231 chars | roles=['assistant', 'system', 'user'] | cats=['language_switching', 'multi_turn_complex']
  [ 4] 138 msgs | 428,263 chars | roles=['assistant', 'system', 'tool', 'user'] | cats=['language_switching', 'tool_heavy', 'multi_turn_complex']
  [ 5]   2 msgs |  11,065 chars | roles=['system', 'user'] | cats=['dissatisfied']
  [ 6]   1 msgs |   1,556 chars | roles=['user'] | cats=['dissatisfied']
  [ 7]   2 msgs |   4,006 chars | roles=['system', 'user'] | cats=['dissatisfied']
  [ 8]  18 msgs |  31,780 chars | roles=['assistant', 'system', 'tool', 'user'] | cats=['dissatisfied']
  [ 9]   8 msgs |  19,917 chars | roles=['assistant', 'system', 'user'] | cats=['dissatisfied', '

## 2. Serialize conversations to one-row-per-conversation CSV

Anonymizer works on a text column in a CSV. We serialize each conversation into
a single string with `<<<MSG role=... idx=...>>>` delimiters so that entity
detection and substitution stay consistent across the whole conversation.

We filter to the **5 shortest** conversations for a fast demo.

Some message content is a JSON array of `{"text": ..., "type": ...}` objects
(e.g. tool-use payloads). We extract and concatenate the `text` fields.

In [3]:
def extract_text(content):
    """Extract plain text from message content (string or JSON-array form)."""
    if content is None:
        return ""
    if isinstance(content, str):
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                parts = [item["text"] for item in parsed if isinstance(item, dict) and "text" in item]
                return "\n".join(parts) if parts else content
        except (json.JSONDecodeError, TypeError):
            pass
        return content
    return str(content)

In [4]:
MSG_DELIM = "<<<MSG role={role} idx={idx}>>>"

def serialize_conversation(conv):
    """Join all messages into a single string with structured delimiters."""
    parts = []
    for idx, msg in enumerate(conv["messages"]):
        text = extract_text(msg.get("content"))
        header = MSG_DELIM.format(role=msg["role"], idx=idx)
        parts.append(f"{header}\n{text}")
    return "\n\n".join(parts)

conv_sizes = [
    (sum(len(m.get("content") or "") for m in c["messages"]), i, c)
    for i, c in enumerate(conversations)
]
conv_sizes.sort()
selected = conv_sizes[:5]

print("Selected 5 shortest conversations:")
for chars, orig_idx, conv in selected:
    n = len(conv["messages"])
    print(f"  orig[{orig_idx:2d}] {n:3d} msgs | {chars:>6,} chars | cats={conv.get('categories', [])}")

rows = []
for _, orig_idx, conv in selected:
    rows.append({
        "conversation_id": conv["conversation_id"],
        "orig_idx": orig_idx,
        "text": serialize_conversation(conv),
    })

conv_df = pd.DataFrame(rows)
print(f"\n{len(conv_df)} rows, text lengths: "
      f"{conv_df['text'].str.len().min():,} - {conv_df['text'].str.len().max():,} chars")

Selected 5 shortest conversations:
  orig[11]   5 msgs |    890 chars | cats=['dissatisfied', 'language_switching']
  orig[ 6]   1 msgs |  1,556 chars | cats=['dissatisfied']
  orig[ 1]   2 msgs |  2,309 chars | cats=['dissatisfied']
  orig[ 7]   2 msgs |  4,006 chars | cats=['dissatisfied']
  orig[ 2]   3 msgs | 10,028 chars | cats=['dissatisfied']

5 rows, text lengths: 1,038 - 10,114 chars


In [5]:
CSV_PATH = Path("_temp_conversations.csv")
conv_df.to_csv(CSV_PATH, index=False)
print(f"Wrote {len(conv_df)} rows to {CSV_PATH}")

Wrote 5 rows to _temp_conversations.csv


## 3. Run Anonymizer -- Substitute mode

Substitute replaces each detected entity with an LLM-generated synthetic value
that fits the context (e.g. matching names to emails, cities to countries).

We start with a preview on a small slice to verify detection quality,
then run the full dataset.

In [6]:
from anonymizer import (
    DEFAULT_ENTITY_LABELS,
    Anonymizer,
    AnonymizerConfig,
    AnonymizerInput,
    Detect,
    Substitute,
)

In [7]:
anonymizer = Anonymizer()

[13:45:01] [INFO] 🔧 Anonymizer initialized with 3 model configs
[13:45:01] [INFO]   |-- 🔎 detector:  gliner-pii-detector
[13:45:01] [INFO]   |-- ✅ validator: gpt-oss-120b
[13:45:01] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


In [13]:
input_data = AnonymizerInput(
    source=str(CSV_PATH),
    text_column="text",
    data_summary=(
        "Multi-turn LLM conversation logs serialized as one text per conversation. "
        "Contains system prompts, user queries, assistant replies, and tool outputs. "
        "May include names, phone/WhatsApp/Telegram numbers, emails, physical addresses, "
        "API keys, user IDs, and file paths with usernames."
    ),
)

config = AnonymizerConfig(
    detect=Detect(),
    replace=Substitute(instructions=""),
)

### Preview on a small sample

In [14]:
preview = anonymizer.preview(config=config, data=input_data, num_records=5)

[13:48:53] [INFO] 📂 Loaded 5 records from _temp_conversations.csv (column: 'text')
[13:48:53] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)
[13:48:53] [INFO]   |-- 👀 Preview mode: processing 5 of 5 records
[13:48:53] [INFO] 🔍 Running entity detection on 5 records
[13:49:38] [INFO]   |-- 📋 Detection complete — 64 entities found across 5 records (0 failed) [44.7s]
[13:49:38] [INFO]   |-- labels: company_name=35, date_time=10, language=7, first_name=4, occupation=4, url=2, user_name=1, last_name=1
[13:49:38] [INFO] 🔄 Running Substitute replacement
[13:49:53] [INFO]   |-- 📋 Replacement complete (0 failed) [15.0s]
[13:49:53] [INFO] 🎉 Pipeline complete — 5 records processed, 0 total failures


In [16]:
preview.display_record(1)

Original,Label,Replacement
2026-01-20T05:32:44.732Z,date_time,2026-02-15T08:12:30.123Z
2026-01-20T05:35:43.000Z,date_time,2026-02-15T08:13:45.000Z
2026-01-20T05:35:53.000Z,date_time,2026-02-15T08:14:10.000Z
2026-01-20T05:51:54.995Z,date_time,2026-02-15T09:20:05.555Z
2026-01-20T05:51:58.763Z,date_time,2026-02-15T09:20:55.777Z
2026-01-20T05:52:44.896Z,date_time,2026-02-15T09:21:30.888Z
2026-01-29T06:29:57.000Z,date_time,2026-02-24T10:05:12.000Z
2026-01-29T06:30:27.000Z,date_time,2026-02-24T10:05:45.500Z
2026-01-29T06:31:01.000Z,date_time,2026-02-24T10:06:10.250Z
2026-01-29T06:31:30.000Z,date_time,2026-02-24T10:06:45.750Z


In [17]:
preview.display_record(2)

Original,Label,Replacement
Alpine.js,company_name,CanyonTech


In [18]:
preview.display_record(3)

Original,Label,Replacement
AI Factory,company_name,Quantum Forge
Huang,last_name,Kumar
Jensen,first_name,Maya
NVIDIA,company_name,Solaris Systems
T5T,company_name,PulseX
TESA,company_name,LUMA
Tesa,company_name,Luma
managers,occupation,logistics officers
writer,occupation,analyst


In [20]:
preview.display_record(4)

Original,Label,Replacement
Bengali (bn),language,Marathi (mr)
English,language,Spanish
English (en),language,French (fr)
Hindi (hi),language,Gujarati (gu)
KFC India,company_name,Domino's India
KFC India Customer Service Agent,occupation,Domino's India Support Representative
Tamil (ta),language,Malayalam (ml)
Telugu (te),language,Kannada (kn)


In [21]:
preview.dataframe

,text,text_with_spans,final_entities,text_replaced
0,<<<MSG role=user idx=0>>>\n안녕 너는 qwen-3 짝퉁이니?...,<<<MSG role=user idx=0>>>\n안녕 너는 [[qwen-3|user...,"{'entities': [{'end_position': 38, 'id': 'user...",<<<MSG role=user idx=0>>>\n안녕 너는 stacy.flynn ...
1,<<<MSG role=user idx=0>>>\n\nYou are an AI ass...,<<<MSG role=user idx=0>>>\n\nYou are an AI ass...,"{'entities': [{'end_position': 665, 'id': 'dat...",<<<MSG role=user idx=0>>>\n\nYou are an AI ass...
2,<<<MSG role=system idx=0>>>\n# Memory Keyword ...,<<<MSG role=system idx=0>>>\n# Memory Keyword ...,"{'entities': [{'end_position': 1850, 'id': 'co...",<<<MSG role=system idx=0>>>\n# Memory Keyword ...
3,<<<MSG role=system idx=0>>>\n# Intent Classifi...,<<<MSG role=system idx=0>>>\n# Intent Classifi...,"{'entities': [{'end_position': 294, 'id': 'com...",<<<MSG role=system idx=0>>>\n# Intent Classifi...
4,<<<MSG role=system idx=0>>>\n# KFC India Custo...,<<<MSG role=system idx=0>>>\n# [[KFC India Cus...,"{'entities': [{'end_position': 62, 'id': 'occu...",<<<MSG role=system idx=0>>>\n# Domino's India ...


## 4. Inspect detected entities

In [ ]:
from collections import Counter

trace = preview.trace_dataframe

label_counts = Counter()
for entities in trace["_detected_entities"]:
    for e in entities:
        label_counts[e["label"]] += 1

print("Detected entity labels (preview):")
for label, count in label_counts.most_common():
    print(f"  {label}: {count}")

In [ ]:
for i in range(min(3, len(trace))):
    entities = trace.loc[i, "_detected_entities"]
    cid = trace.loc[i, "conversation_id"][:8]
    print(f"--- conv {i} | {cid}... | {len(entities)} entities ---")
    for e in entities[:15]:
        print(f"  [{e['label']}] {e['value']!r}")
    if len(entities) > 15:
        print(f"  ... and {len(entities) - 15} more")
    print()

## 5. Full run

In [ ]:
result = anonymizer.run(config=config, data=input_data)
print(result)

In [ ]:
result.display_record()

In [ ]:
result.dataframe.head()

### Entity summary across the full dataset

In [ ]:
full_trace = result.trace_dataframe

full_label_counts = Counter()
for entities in full_trace["_detected_entities"]:
    for e in entities:
        full_label_counts[e["label"]] += 1

print(f"Total entities detected: {sum(full_label_counts.values())}")
print(f"Unique labels: {len(full_label_counts)}")
print()
for label, count in full_label_counts.most_common():
    print(f"  {label}: {count}")

## 6. Reassemble into conversation format

Split the substituted text on the `<<<MSG ...>>>` delimiters to recover individual
messages, then plug them back into the original conversation structure.

In [ ]:
import copy
import re

DELIM_PATTERN = re.compile(r"<<<MSG role=(\w+) idx=(\d+)>>>")

def split_scrubbed_text(scrubbed_text):
    """Split serialized conversation back into {idx: content} mapping."""
    parts = DELIM_PATTERN.split(scrubbed_text)
    # parts = [preamble, role0, idx0, content0, role1, idx1, content1, ...]
    messages = {}
    i = 1
    while i + 2 < len(parts):
        idx = int(parts[i + 1])
        content = parts[i + 2].strip()
        messages[idx] = content
        i += 3
    return messages

result_df = result.dataframe
replaced_col = "text_replaced"

scrubbed_conversations = []
for _, row in result_df.iterrows():
    cid = row["conversation_id"]
    orig_conv = next(c for c in conversations if c["conversation_id"] == cid)
    new_conv = copy.deepcopy(orig_conv)

    msg_map = split_scrubbed_text(row[replaced_col])
    for idx, content in msg_map.items():
        if idx < len(new_conv["messages"]):
            new_conv["messages"][idx]["content"] = content
    scrubbed_conversations.append(new_conv)

print(f"Reassembled {len(scrubbed_conversations)} conversations")

In [ ]:
OUTPUT_PATH = Path("reduced.scrubbed.jsonl")
with open(OUTPUT_PATH, "w") as f:
    for conv in scrubbed_conversations:
        f.write(json.dumps(conv, ensure_ascii=False) + "\n")

print(f"Wrote {len(scrubbed_conversations)} conversations to {OUTPUT_PATH}")

### Quick diff: original vs scrubbed

In [ ]:
for conv_pos, scrub in enumerate(scrubbed_conversations):
    cid = scrub["conversation_id"]
    orig = next(c for c in conversations if c["conversation_id"] == cid)

    print(f"\nConversation {conv_pos} ({orig.get('categories', [])})")
    print("=" * 60)
    for i, (om, sm) in enumerate(zip(orig["messages"], scrub["messages"])):
        oc = (om.get("content") or "")[:300]
        sc = (sm.get("content") or "")[:300]
        if oc != sc:
            print(f"\n  msg[{i}] role={om['role']} ** CHANGED **")
            print(f"    ORIG:  {oc[:200]}...")
            print(f"    SCRUB: {sc[:200]}...")
        else:
            print(f"  msg[{i}] role={om['role']} -- unchanged")

In [ ]:
CSV_PATH.unlink(missing_ok=True)
print("Cleaned up temp CSV.")\n